# 02 – Link postcodes to buurten (neighbourhoods)

**Goal:** link every Liander postcode row to a CBS buurt (2023 boundaries) and aggregate to one row per buurt per year.
**Inputs:** `pdok_pc6_centroids_liander.csv`, `cbs_buurten_2023_simplified.geojson`, `liander_electricity_2019_2026.csv`
**Outputs:** `Data/processed/pc6_to_buurt_2023.csv`, `Data/processed/buurt_year_electricity.csv`

## Method
- Water buurten and "Buitenland" removed (no homes there).
- Spatial join: postcode centre point **within** buurt polygon → 99.94% matched.
- 29 points on a border (2 buurten) → first kept. 113 unmatched → nearest buurt within 500 m (110 matched, median distance 7.9 m, max 407 m); 3 dropped.
- Result: 189,062 postcodes with exactly one buurt (`method` = within / nearest).

## Linking Liander rows
- Each row is a postcode **range**; the buurt of the start postcode is used, falling back to the end postcode.
- kWh linked to a buurt: 99.5% (2019–2024), 99.98% (2025–2026). Missing = mostly retired postcodes.
- 6–7.5% of kWh comes from ranges that **cross a buurt border** → assigned to the start buurt (approximation, stable over the years). Tracked per buurt in `share_cross_border_pct`.

## Aggregation (per buurt per year)
- Connections and kWh summed; percentages averaged **weighted by connections**.
- `share_3phase_pct` = electrification indicator (connections in rows whose most common type is 3x…).
- Smart meter, consume-only and low-tariff % do not exist in 2025–2026 → left empty (NaN), not 0.
- Result: ~5,600 buurten per year, 44,762 buurt-year rows.

## Limitations
- Buurt boundaries of 2023 are used for all years.
- `SOORT_AANSLUITING` is the most common type in a range, not the type of every connection.

In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

RAW = Path("../Data/raw")
PROCESSED = Path("../Data/processed")
EXT = RAW / "External"

# postcode centre points
pc6 = pd.read_csv(EXT / "pdok_pc6_centroids_liander.csv")
print("POSTCODES:", pc6.shape)
print(pc6.head(3))

# buurt boundaries (shapes)
buurten = gpd.read_file(EXT / "cbs_buurten_2023_simplified.geojson")
print("\nBUURTEN:", buurten.shape)
print("Coordinate system:", buurten.crs)
print("Columns:", buurten.columns.tolist())
print(buurten.drop(columns="geometry").head(3))

POSTCODES: (189065, 6)
      pc6       lon        lat  n_streets  gemeentecode woonplaats
0  1011KM  4.896560  52.367797          1           363  Amsterdam
1  1011KN  4.896929  52.367825          1           363  Amsterdam
2  1011TG  4.907475  52.372121          1           363  Amsterdam

BUURTEN: (14515, 11)
Coordinate system: EPSG:4326
Columns: ['buurtcode', 'buurtnaam', 'wijkcode', 'gemeentecode', 'gemeentenaam', 'water', 'meest_voorkomende_postcode', 'aantal_inwoners', 'woningvoorraad', 'oppervlakte_land_in_ha', 'geometry']
    buurtcode          buurtnaam  wijkcode gemeentecode gemeentenaam water  \
0  BU09989999         Buitenland  WK099899       GM0998   Buitenland     B   
1  BU00349997  Groot binnenwater  WK003499       GM0034       Almere    JA   
2  BU00509997  Groot binnenwater  WK005099       GM0050     Zeewolde    JA   

  meest_voorkomende_postcode  aantal_inwoners  woningvoorraad  \
0                     -99997           -99997          -99997   
1                    

In [2]:
# 1. keep only land buurten (no water, no "Buitenland")
land = buurten[(buurten["water"] != "JA") & (buurten["gemeentenaam"] != "Buitenland")]
land = land[["buurtcode", "buurtnaam", "wijkcode", "gemeentecode", "gemeentenaam", "geometry"]]
print("Land buurten:", len(land))

# 2. turn the postcode table into points on the map
pc6_points = gpd.GeoDataFrame(
    pc6,
    geometry=gpd.points_from_xy(pc6["lon"], pc6["lat"]),
    crs="EPSG:4326",
)

# 3. spatial join: which buurt contains each point?
joined = gpd.sjoin(pc6_points, land, how="left", predicate="within")

matched = joined["buurtcode"].notna().sum()
print(f"Postcodes matched to a buurt: {matched:,} of {len(joined):,} "
      f"({100 * matched / len(joined):.2f}%)")
print("Not matched:", joined["buurtcode"].isna().sum())
print("Duplicates (point in 2 buurten):", joined["pc6"].duplicated().sum())

Land buurten: 14421
Postcodes matched to a buurt: 188,981 of 189,094 (99.94%)
Not matched: 113
Duplicates (point in 2 buurten): 29


In [3]:
# column names after the join: the buurt's municipality is "gemeentecode_right"
cols = ["pc6", "lon", "lat", "woonplaats",
        "buurtcode", "buurtnaam", "wijkcode", "gemeentecode_right", "gemeentenaam"]

# 1. duplicates: keep the first buurt for each postcode
joined = joined.drop_duplicates(subset="pc6", keep="first")

ok = joined[joined["buurtcode"].notna()][cols].copy()
ok["method"] = "within"

# 2. unmatched: take the NEAREST buurt, but only if it is within 500 metres
missing = joined[joined["buurtcode"].isna()][["pc6", "lon", "lat", "woonplaats", "geometry"]]
missing_rd = missing.to_crs("EPSG:28992")       # Dutch grid in metres
land_rd = land.to_crs("EPSG:28992")
near = gpd.sjoin_nearest(missing_rd, land_rd, how="left",
                         max_distance=500, distance_col="dist_m")
near = near.drop_duplicates(subset="pc6", keep="first")
print("Nearest-buurt distance (m):")
print(near["dist_m"].describe().round(1))

near = near[near["buurtcode"].notna()].rename(columns={"gemeentecode": "gemeentecode_right"})
near = near[cols].copy()
near["method"] = "nearest"

# 3. combine and check
pc6_buurt = pd.concat([ok, near], ignore_index=True)
pc6_buurt = pc6_buurt.rename(columns={"gemeentecode_right": "gemeentecode"})

print("\nFinal postcodes:", len(pc6_buurt), "of", len(pc6))
print("Duplicates:", pc6_buurt["pc6"].duplicated().sum())
print(pc6_buurt["method"].value_counts())

# 4. save
pc6_buurt.to_csv(PROCESSED / "pc6_to_buurt_2023.csv", index=False)
print("\nSaved: pc6_to_buurt_2023.csv")

Nearest-buurt distance (m):
count    110.0
mean      32.7
std       60.7
min        0.0
25%        1.6
50%        7.9
75%       37.4
max      406.6
Name: dist_m, dtype: float64

Final postcodes: 189062 of 189065
Duplicates: 0
method
within     188952
nearest       110
Name: count, dtype: int64

Saved: pc6_to_buurt_2023.csv


In [4]:
elec = pd.read_csv(PROCESSED / "liander_electricity_2019_2026.csv")
print("Liander rows:", elec.shape)

lookup = pc6_buurt.set_index("pc6")["buurtcode"]

elec["buurt_van"] = elec["POSTCODE_VAN"].map(lookup)
elec["buurt_tot"] = elec["POSTCODE_TOT"].map(lookup)

print(f"Start postcode found: {elec['buurt_van'].notna().mean() * 100:.2f}%")
print(f"End postcode found:   {elec['buurt_tot'].notna().mean() * 100:.2f}%")

both = elec["buurt_van"].notna() & elec["buurt_tot"].notna()
same = (elec.loc[both, "buurt_van"] == elec.loc[both, "buurt_tot"]).mean()
print(f"Start and end in the SAME buurt: {same * 100:.2f}%")

print("\nFound per year (% of rows with a buurt for the start postcode):")
print((elec.groupby("year")["buurt_van"].apply(lambda s: s.notna().mean() * 100)).round(2))

Liander rows: (1167812, 12)
Start postcode found: 98.82%
End postcode found:   99.35%
Start and end in the SAME buurt: 96.56%

Found per year (% of rows with a buurt for the start postcode):
year
2019    98.55
2020    98.55
2021    98.55
2022    98.56
2023    98.55
2024    98.54
2025    99.59
2026    99.61
Name: buurt_van, dtype: float64


In [5]:
# use the start postcode; if unknown, fall back to the end postcode
elec["buurtcode"] = elec["buurt_van"].fillna(elec["buurt_tot"])
elec["crosses_border"] = both & (elec["buurt_van"] != elec["buurt_tot"])

# coverage in kWh per year
cov = elec.groupby("year").apply(lambda d: pd.Series({
    "rows_with_buurt_pct": d["buurtcode"].notna().mean() * 100,
    "kwh_with_buurt_pct": d.loc[d["buurtcode"].notna(), "total_kwh"].sum() / d["total_kwh"].sum() * 100,
    "kwh_crossing_border_pct": d.loc[d["crosses_border"], "total_kwh"].sum() / d["total_kwh"].sum() * 100,
})).round(2)
print(cov)

      rows_with_buurt_pct  kwh_with_buurt_pct  kwh_crossing_border_pct
year                                                                  
2019                99.18               99.50                     7.53
2020                99.19               99.51                     7.55
2021                99.20               99.52                     7.41
2022                99.20               99.53                     7.29
2023                99.20               99.52                     7.21
2024                99.20               99.49                     6.31
2025                99.97               99.96                     6.33
2026                99.99               99.98                     6.29


C:\Users\Karam\AppData\Local\Temp\ipykernel_20260\2839763667.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cov = elec.groupby("year").apply(lambda d: pd.Series({


In [6]:
e = elec[elec["buurtcode"].notna()].copy()
w = e["AANSLUITINGEN_AANTAL"]

# helper columns: percentages x connections (so we can compute weighted averages)
e["w_smart"] = e["SLIMME_METER_PERC"] * w
e["w_consume_only"] = e["LEVERINGSRICHTING_PERC"] * w
e["w_low_tariff"] = e["SJV_LAAG_TARIEF_PERC"] * w
e["conn_3phase"] = w.where(e["SOORT_AANSLUITING"].astype(str).str.startswith("3x"), 0)
e["conn_cross"] = w.where(e["crosses_border"], 0)

bu = e.groupby(["buurtcode", "year", "series"]).agg(
    n_rows=("POSTCODE_VAN", "size"),
    connections=("AANSLUITINGEN_AANTAL", "sum"),
    total_kwh=("total_kwh", "sum"),
    w_smart=("w_smart", "sum"),
    w_consume_only=("w_consume_only", "sum"),
    w_low_tariff=("w_low_tariff", "sum"),
    conn_3phase=("conn_3phase", "sum"),
    conn_cross=("conn_cross", "sum"),
).reset_index()

# final indicators per buurt-year
bu["kwh_per_conn"] = bu["total_kwh"] / bu["connections"]
bu["smart_meter_pct"] = bu["w_smart"] / bu["connections"]
bu["consume_only_pct"] = bu["w_consume_only"] / bu["connections"]
bu["low_tariff_pct"] = bu["w_low_tariff"] / bu["connections"]
bu["share_3phase_pct"] = 100 * bu["conn_3phase"] / bu["connections"]
bu["share_cross_border_pct"] = 100 * bu["conn_cross"] / bu["connections"]

# these three columns do not exist in the new 2025-2026 files
new = bu["series"] == "new_2025_2026"
bu.loc[new, ["smart_meter_pct", "consume_only_pct", "low_tariff_pct"]] = None

bu = bu.drop(columns=["w_smart", "w_consume_only", "w_low_tariff", "conn_3phase", "conn_cross"])

print("Shape:", bu.shape)
print("\nBuurten per year:")
print(bu.groupby("year")["buurtcode"].nunique())
print("\nExample rows:")
print(bu.head(3).round(1))

bu.to_csv(PROCESSED / "buurt_year_electricity.csv", index=False)
print("\nSaved: buurt_year_electricity.csv")

Shape: (44762, 12)

Buurten per year:
year
2019    5558
2020    5568
2021    5577
2022    5589
2023    5602
2024    5631
2025    5617
2026    5620
Name: buurtcode, dtype: int64

Example rows:
    buurtcode  year         series  n_rows  connections  total_kwh  \
0  BU00340101  2019  old_2019_2024      19        424.0  2974736.0   
1  BU00340101  2020  old_2019_2024      20        447.0  2721649.0   
2  BU00340101  2021  old_2019_2024      20        450.0  2697549.0   

   kwh_per_conn  smart_meter_pct  consume_only_pct  low_tariff_pct  \
0        7015.9             69.3              96.9            81.1   
1        6088.7             70.2              96.6            78.7   
2        5994.6             71.8              96.2            80.2   

   share_3phase_pct  share_cross_border_pct  
0               4.7                    26.9  
1               4.5                    26.2  
2               4.4                    26.2  

Saved: buurt_year_electricity.csv
